In [ ]:
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from csim.simulation import Simulator
from csim.entities import Spacecraft
from csim.world import MU_EARTH, R_EARTH
from csim.math.time import greg_to_jd
from csim.math.coes import coes_to_rv
from csim.math.transformations import itrf_to_gcrs_matrices, R_matrix, ECEF_to_geodetic_LLA
from csim.constants import ARCSEC_TO_RAD, SEC_TO_DAY, DEG_TO_RAD
from csim.visualization import plot_orbit

In [ ]:
r = R_EARTH + 35786e3  # GEO

v = np.sqrt(MU_EARTH / r)
print(r)
print(v)

In [ ]:
dt = 1
t0 = 0
n_steps = 86400  # About 2 days
i = 8 * np.pi / 180
state0 = np.array([r * np.cos(i), 0, r * np.sin(i), 0, v, 0, 1, 0, 0, 0, 1, 0, 0])

sc = Spacecraft(100, None)
sim = Simulator(state0, t0, dt, n_steps, sc)

sim.simulate()

In [ ]:
n = -1
plt.plot(sim.X[:n, 0], sim.X[:n, 1])
plt.axis("equal")
plt.xlim()
plt.show()

In [ ]:
plt.plot(sim.X[:, 0])

In [ ]:
plt.plot(sim.w_meas[:, 0])
plt.show()
plt.plot(sim.a_meas[:, 0])

In [ ]:
plot_orbit(sim.X)

In [ ]:
jd = greg_to_jd(2004, 4, 6, 7, 51, 28.386009)

xp = -0.140682 * ARCSEC_TO_RAD
yp = 0.333309 * ARCSEC_TO_RAD
dAT = 32
dUT1 = 0.4399619
dX = -0.000205 * ARCSEC_TO_RAD
dY = -0.000136 * ARCSEC_TO_RAD

ECEF = []

PN, _, W = itrf_to_gcrs_matrices(xp, yp, jd, dX, dY, dUT1, dAT)

# jd_utc + dUT1 * SEC_TO_DAY

for row in sim.X[:, :3]:  # 27 per second :/
    # PN, R, W = itrf_to_gcrs_matrices(xp, yp, jd, dX, dY, dUT1, dAT)
    R = R_matrix(jd + dUT1 * SEC_TO_DAY)
    ECEF.append((PN @ R @ W).T @ row)
    # Increment jd
    jd += dt / 86400
ECEF = np.asarray(ECEF)

In [ ]:
# plt.plot(ECEF[:,0])

plot_orbit(ECEF)

In [ ]:
track = np.array([ECEF_to_geodetic_LLA(r) for r in ECEF])

In [ ]:
plt.plot(track[:, 1], track[:, 0], ".", markersize=1)

In [ ]:
dt = 1
t0 = 0

n_steps = int(800 * 60 * 2 / dt)

r_vec, v_vec = coes_to_rv(
    26600e3, 0.74, 63.4 * DEG_TO_RAD, 0, 270 * DEG_TO_RAD, 0, MU_EARTH
)
# print(rv)
state0 = np.hstack((r_vec, -v_vec, [1, 0, 0, 0, 0, 0, 0]))

sc = Spacecraft(100, None)
sim = Simulator(state0, t0, dt, n_steps, sc)

sim.simulate()

In [ ]:
plot_orbit(sim.X)

In [ ]:
jd = greg_to_jd(2004, 4, 6, 7, 51, 28.386009)

xp = -0.140682 * ARCSEC_TO_RAD
yp = 0.333309 * ARCSEC_TO_RAD
dAT = 32
dUT1 = 0.4399619
dX = -0.000205 * ARCSEC_TO_RAD
dY = -0.000136 * ARCSEC_TO_RAD

ECEF = []

PN, _, W = itrf_to_gcrs_matrices(xp, yp, jd, dX, dY, dUT1, dAT)

# jd_utc + dUT1 * SEC_TO_DAY

for row in sim.X[:, :3]:  # 27 per second :/
    # PN, R, W = itrf_to_gcrs_matrices(xp, yp, jd, dX, dY, dUT1, dAT)
    R = R_matrix(jd + dUT1 * SEC_TO_DAY)
    ECEF.append(PN @ R @ W @ row)
    # Increment jd
    jd += dt / 86400
ECEF = np.asarray(ECEF)

In [ ]:
plot_orbit(ECEF)

In [ ]:
track = np.array([ECEF_to_geodetic_LLA(r) for r in ECEF])

In [ ]:
plt.plot(track[:, 1], track[:, 0], ".", markersize=1)
plt.ylabel("Latitude (deg)")
plt.xlabel("Longitude (deg)")

# 

In [ ]:
dt = 0.01
t0 = 0
n_steps = 5000

r = R_EARTH
state0 = np.array([0, 0, R_EARTH, 2, 0, 300, 1, 0, 0, 0, 0, 0, 0])

sc = Spacecraft(100, None)
sim2 = Simulator(state0, t0, dt, n_steps, sc)

sim2.simulate()

In [ ]:
# plt.plot(sim2.X[:,0])
# plt.show()
# plt.plot(sim2.X[:,1])
# plt.show()
# plt.plot(sim2.X[:,2])
# plt.show()

In [ ]:
plot_orbit(sim2.X)

In [ ]:
# plot orbits
fig = go.Figure()

# Plot trajectory
fig.add_scatter3d(
    x=sim2.X[:, 0],
    y=sim2.X[:, 1],
    z=sim2.X[:, 2],
    mode="markers",
    marker=dict(size=1, color="blue"),
    name="Trajectory",
)

# Final Position
fig.add_scatter3d(
    x=[sim2.X[-1, 0]],
    y=[sim2.X[-1, 1]],
    z=[sim2.X[-1, 2]],
    mode="markers",
    marker=dict(size=10, color="green"),
    name="Final Position",
)


# Initial Position
fig.add_scatter3d(
    x=[sim2.X[0, 0]],
    y=[sim2.X[0, 1]],
    z=[sim2.X[0, 2]],
    mode="markers",
    marker=dict(
        size=10,
        color="red",
    ),
    name="Initial Position",
)

# Temp Earth

x = [[100, 100], [0, 0]]

y = [[1, -1], [1, -1]]

z = [[R_EARTH, R_EARTH], [R_EARTH, R_EARTH]]

fig.add_surface(
    x=x,
    y=y,
    z=z,
    colorscale=[[0, "orange"], [1, "orange"]],
    showlegend=False,
    showscale=False,
    name="Earth",
)

fig.update_layout(
    width=1000,
    height=800,
    title_font=dict(size=24, family="Garamond"),
    title_text="Rocket Trajectory",
    title_x=0.5,
    title_y=0.9,
    scene=dict(
        # xaxis = dict(nticks=4, range=[-100,100]),
        # yaxis = dict(nticks=4, range=[-50,100]),
        zaxis=dict(nticks=4, range=[R_EARTH - 10, R_EARTH + 6000])
    ),
)
fig.show()

In [ ]:
plt.plot(sim2.X[:, 0], sim2.X[:, 1])
plt.xlim()
plt.show()